# Doppl Lineage — Neo4j Query Spike

**Throwaway** — never imported by runtime. See `README.md`.

Load a derived `LineageGraphProjection` export and prove four query shapes.

In [ ]:
# Connect to local Neo4j (start it via the README's docker command first).
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "spike-password"))
EXPORT_PATH = "lineage-<runId>.json"  # ← replace with your actual export filename

In [ ]:
# Load the JSON export into Neo4j via APOC.
LOAD_CYPHER = """
CALL apoc.load.json($path) YIELD value
UNWIND value.nodes AS n
MERGE (node:Node {id: n.id})
SET node.type = n.type, node.label = n.label, node.status = n.status
WITH value
UNWIND value.edges AS e
MATCH (s:Node {id: e.source}), (t:Node {id: e.target})
MERGE (s)-[r:RELATES {type: e.type}]->(t)
SET r.label = e.label
"""
with driver.session() as session:
    session.run(LOAD_CYPHER, path=EXPORT_PATH)
print("loaded")

## Query 1: Ancestors of the selected winner

Walk parent edges from a winning candidate back to gen-0 agenomes.

In [ ]:
Q1 = """
MATCH path = (winner:Node {type: 'candidate', status: 'selected'})
             <-[:RELATES*]-(ancestor:Node {type: 'agenome'})
RETURN ancestor.id AS id, length(path) AS distance
ORDER BY distance ASC
"""
with driver.session() as session:
    for record in session.run(Q1):
        print(record['id'], record['distance'])

## Query 2: Parent contribution

For each top-K agenome, count descendants that produced a candidate.

In [ ]:
Q2 = """
MATCH (p:Node {type: 'agenome'})-[:RELATES*]->(c:Node {type: 'candidate'})
RETURN p.id AS agenome, count(DISTINCT c) AS candidates
ORDER BY candidates DESC
"""
with driver.session() as session:
    for record in session.run(Q2):
        print(record['agenome'], record['candidates'])

## Query 3: Critic-kill patterns

Which critic mandates drove the most cull events.

In [ ]:
Q3 = """
MATCH (r:Node {type: 'critic_review'})-[:RELATES]->(c:Node {type: 'candidate'})
WHERE c.status = 'invalid' OR c.status = 'culled'
RETURN r.label AS mandate, count(*) AS kills
ORDER BY kills DESC
"""
with driver.session() as session:
    for record in session.run(Q3):
        print(record['mandate'], record['kills'])

## Query 4: Lineage distance / diversity

Average path length over the candidate subgraph.

In [ ]:
Q4 = """
MATCH (a:Node {type: 'agenome'})-[:RELATES*]->(c:Node {type: 'candidate'})
WITH avg(size(((a)-[:RELATES*]-(c)))) AS avg_steps
RETURN avg_steps
"""
with driver.session() as session:
    for record in session.run(Q4):
        print(record['avg_steps'])

In [ ]:
driver.close()